In [ ]:
# --timeframe 1d   : Таймфрейм свечей (дневные данные).
# --start-year 2000: Глубина загрузки истории (начиная с 2000 года).
# --workers 6      : Количество параллельных потоков для ускорения загрузки.

!python -m _tools.update_market_data --timeframe 1d --start-year 2000 --workers 6
!python -m _tools.update_macro --timeframe 1d --start-year 2000 --workers 6
# Выполняет комплексную проверку целостности, отсутствия пропусков и корректности OHLCV данных.
!python -m _tools.check_data_quality

In [1]:
# --timeframe 1d       : Интервал данных — дневные свечи.
# --lookback 60        : Глубина истории — модель смотрит на 60 дней назад.
# --horizon 10         : Горизонт прогноза — ищем выход по барьерам в течение 10 дней.
# --auto               : Режим автоматического расчета уровней TP/SL на основе волатильности.
# --percentile 75      : Перцентиль волатильности для отсечения аномальных выбросов при авто-разметке.
# --init_split         : Дата начала первого разделения данных на Train и Val.
# --val_interval 2     : Продолжительность валидационного периода в годах.
# --split_interval 2   : Шаг смещения окна Walk-Forward в годах.
# --endpoint           : Дата окончания формирования всех временных интервалов.
# --corr_threshold     : Порог удаления коррелирующих признаков (убираем дубликаты > 85%).
# --cum_threshold      : Порог кумулятивной важности (оставляем топ фичей, дающих 99% влияния).
# --force              : Раскомментируйте параметр ниже для полной перезаписи кэшированных данных.

!python -m _tools.init_dataset \
    --timeframe 1d \
    --lookback 18 \
    --horizon 3 \
    --auto \
    --percentile 75 \
    --init_split 2010-01-01 \
    --val_interval 2 \
    --split_interval 2 \
    --endpoint 2024-01-01 \
    --corr_threshold 0.85 \
    --cum_threshold 0.99 \
    #--force


🧹 Запуск модуля очистки данных (Сплиты, Иглы, Выбросы)...
Корректировка сплитов:  13%|██▊                  | 9/68 [00:00<00:00, 82.53it/s]  🕵️‍♂️ [HEURISTIC SPLIT] LSNGP@MISX на 2005-08-03: Коэфф 5.0
  📌 [KNOWN SPLIT] TRNFP@MISX на 2024-02-21: Коэфф 0.01
Корректировка сплитов: 100%|███████████████████| 68/68 [00:00<00:00, 100.86it/s]
✅ Очистка завершена!

🔄 [fold_2010/train - Build] Запуск...
✅ Готово: сохранено в /home/restorator/trader_test/data/processed/2000_2026_1d_18_3/fold_2010/data/train/dataset.csv

🔄 [fold_2010/train - Labels] Запуск...
✅ Авто-уровни: TP=5.94%, SL=5.61%
Разметка: 100%|████████████████████████████████| 39/39 [00:00<00:00, 230.00it/s]
🎉 Размеченный датасет сохранен: labels.csv

🔄 [fold_2010/train - Features] Запуск...

⚙️ [TRAIN] Инициализация расчета...
Этап 1: Расчет кросс-секционных признаков...
Индивидуальные фичи: 100%|██████████████████████| 33/33 [00:33<00:00,  1.01s/it]
✂️ Winsorization (0.1% - 99.9%) для 149 признаков...
📈 Обучение Scaler...
💾 Сохране

In [ ]:
#полная проверка подготовленных на предыдущем этапе данных
!python -m _tools.verify_data

In [3]:
!python run_walkforward.py \
    --dataset_dir "data/processed/2000_2026_1d_18_3" \
    --runs 100 \
    --batch_size 8192 \
    --epochs 100 \
    --l2_reg "1e-4" \
    --lr "1e-3" \
    --start_fold "fold_2010" \
    --append

🚀 Запуск массового обучения моделей (Walk-Forward)...
📁 Датасет: data/processed/2000_2026_1d_18_3
⚙️  Настройки: 100 runs, 100 epochs, batch 8192
⏭️ Пропускаем завершенные фолды. Начинаем строго с: fold_2010

🔥 Обучение нейросети для: fold_2010
✅ Mixed precision включена!
✅ Динамическое выделение видеопамяти включено!
🚀 Старт обучения. Фолд: [fold_2010]
📊 Форма данных: [Lookback: 18, Features: 69]
⚙️ Расчет идеальных весов классов...
   Баланс: SL(0)=8802, Hold(1)=23356, TP(2)=8625
   Веса:   SL(0)=1.54, Hold(1)=0.58, TP(2)=1.58
⏳ Подготовка конвейера данных...

--------------------------------------------------
🔄 ИТЕРАЦИЯ 1/100 (Лучшая точность сессии: 0.00%)
--------------------------------------------------
Epoch 1/100
5/5 - 6s - 1s/step - accuracy: 0.4117 - loss: 1.3309 - val_accuracy: 0.4791 - val_loss: 1.1468
Epoch 2/100
5/5 - 1s - 206ms/step - accuracy: 0.4523 - loss: 1.1633 - val_accuracy: 0.4516 - val_loss: 1.1715
Epoch 3/100
5/5 - 1s - 261ms/step - accuracy: 0.5149 - loss: 1.

In [ ]:
#очистка наименне успешных ltsm моделей (остается топ 3)
!python -m _tools.clean_lstm_models

In [1]:
!python -m _tools.evaluate_lstm_predictions

📊 Загрузка данных из data/processed/2000_2026_1d/rl_env/environment_data.parquet...
❌ Файл не найден!


In [2]:
!python -m _tools.prepare_rl_env

I0000 00:00:1776973973.083640    2341 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776973977.030997    2341 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
❌ Не найден базовый файл котировок data/processed/2000_2026_1d/raw_combined.csv


In [ ]:
!python -m _tools.train_rllib_pbt --population 4 --iterations 3000